<a href="https://colab.research.google.com/github/jrain-dev/CS-7357-NeuralNetsAndDeepLearning-/blob/main/MiniHandsOnExercise1Faces.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [43]:
%pip install skorch torch

In [44]:
import numpy as np
from sklearn.datasets import fetch_olivetti_faces
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import torch
import torch.nn as nn
from skorch import NeuralNetClassifier

In [45]:
def make_mlp(in_dim, layer_sizes):
    layers = []
    for h in layer_sizes:
        layers += [nn.Linear(in_dim, h), nn.ReLU()]
        in_dim = h
    return nn.Sequential(*layers), in_dim


In [46]:
class ParallelMultiPathNet(nn.Module):
    def __init__(self, input_dim, num_classes,
                 pre_layers=(128, 64),
                 paths=((64, 32), (64, 32, 16, 32)),
                 fusion_dim=128,
                 post_layers=(64, 32),
                 fusion_mode="concat"):
        super().__init__()
        if not 1 < len(pre_layers) <= 5:
            raise ValueError("Need 1 < m <= 5 layers before the parallel module")
        if not 2 <= len(paths) <= 3:
            raise ValueError("Need 2 <= k <= 3 parallel paths")
        if not 1 <= len(post_layers) <= 5:
            raise ValueError("Need 1 <= n + l <= 5 layers after fusion")
        if fusion_mode not in ("concat", "add"):
            raise ValueError("fusion_mode must be 'concat' or 'add'")
        if fusion_mode == "add" and len({p[-1] for p in paths}) != 1:
            raise ValueError("For 'add' fusion, all paths must end with the same width")

        self.fusion_mode = fusion_mode
        self.pre, dim = make_mlp(input_dim, pre_layers)

        self.paths = nn.ModuleList()
        out_dims = []
        for sizes in paths:
            path, out_dim = make_mlp(dim, sizes)
            self.paths.append(path)
            out_dims.append(out_dim)

        fused_in = sum(out_dims) if fusion_mode == "concat" else out_dims[0]
        self.fusion = nn.Sequential(nn.Linear(fused_in, fusion_dim), nn.ReLU())
        self.post, dim = make_mlp(fusion_dim, post_layers)
        self.classifier = nn.Linear(dim, num_classes)

    def forward(self, x):
        x = self.pre(x)
        outs = [path(x) for path in self.paths]
        x = torch.cat(outs, dim=1) if self.fusion_mode == "concat" else torch.stack(outs).sum(dim=0)
        return self.classifier(self.post(self.fusion(x)))

In [47]:
# Load dataset
faces = fetch_olivetti_faces()
X = faces.data
y = faces.target
print("Dataset shape:", X.shape)
print("Number of classes:", len(np.unique(y)))
print("Number of input features:", X.shape[1]) # every image has 64 input features

# Initialize training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)
print("=" * 50)
print("Training samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])
print("=" * 50)

# Scale Data
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

downloading Olivetti faces from https://ndownloader.figshare.com/files/5976027 to /root/scikit_learn_data
Dataset shape: (400, 4096)
Number of classes: 40
Number of input features: 4096
Training samples: 320
Testing samples: 80


In [48]:
# Initialize Bottleneck Model (sklearn)
Bottleneck = MLPClassifier(
    hidden_layer_sizes=(256, 128, 64, 128),
    activation="relu",
    solver="adam",
    max_iter=500,
    random_state=42
)

# Initialize Parallel Model (skorch)
Parallel = NeuralNetClassifier(
    ParallelMultiPathNet,
    module__input_dim=X_train.shape[1],
    module__num_classes=len(np.unique(y)),
    module__pre_layers=(128, 64),
    criterion=nn.CrossEntropyLoss,
    optimizer=torch.optim.Adam,
    lr=1e-3,
    max_epochs=50,
    train_split=None,
)

# Initialize Custom Model (sklearn)
Reduction = MLPClassifier(
    hidden_layer_sizes=(256, 128, 64, 32, 16),
    activation="relu",
    solver="adam",
    max_iter=500,
    random_state=42
)

In [49]:
# Train Models
print("=" * 50)
print("Training in process, please wait.")
Bottleneck.fit(X_train_scaled, y_train)
Parallel.fit(
    np.asarray(X_train_scaled, dtype="float32"),
    np.asarray(y_train, dtype="int64"),
)
Reduction.fit(X_train_scaled, y_train)
print("Training complete.")
print("=" * 50)

# Evaluate Models
b_y_pred = Bottleneck.predict(X_test_scaled)
p_y_pred = Parallel.predict(np.asarray(X_test_scaled, dtype="float32"))
r_y_pred = Reduction.predict(X_test_scaled)
print("=" * 50)
print("BOTTLNECK PERFORMANCE")
print("Test Accuracy:", accuracy_score(y_test, b_y_pred))
print("Classification Report:")
print(classification_report(y_test, b_y_pred))
print("Confusion Matrix:")
print(confusion_matrix(y_test, b_y_pred))
print("=" * 50)
print("PARALLEL PERFORMANCE")
print("Test Accuracy:", accuracy_score(y_test, p_y_pred))
print("Classification Report:")
print(classification_report(y_test, p_y_pred))
print("Confusion Matrix:")
print(confusion_matrix(y_test, p_y_pred))
print("=" * 50)
print("REDUCTION PERFORMANCE")
print("Test Accuracy:", accuracy_score(y_test, r_y_pred))
print("Classification Report:")
print(classification_report(y_test, r_y_pred))
print("Confusion Matrix:")
print(confusion_matrix(y_test, r_y_pred))


Training in process, please wait.
  epoch    train_loss     dur
-------  ------------  ------
      1        3.6952  0.1510
      2        3.6853  0.0795
      3        3.6632  0.0760
      4        3.6157  0.0606
      5        3.5241  0.0549
      6        3.3917  0.0958
      7        3.2586  0.0677
      8        3.1213  0.1526
      9        2.9677  0.1302
     10        2.7759  0.1572
     11        2.5601  0.1102
     12        2.3051  0.2032
     13        2.0329  0.2197
     14        1.7815  0.1789
     15        1.5441  0.1583
     16        1.3304  0.1327
     17        1.1367  0.1555
     18        1.0036  0.1854
     19        0.8696  0.1951
     20        0.7302  0.0679
     21        0.6922  0.1465
     22        0.5763  0.1459
     23        0.5133  0.1474
     24        0.4637  0.1193
     25        0.3546  0.1105
     26        0.3378  0.1337
     27        0.3026  0.1355
     28        0.2171  0.2083
     29        0.1912  0.1120
     30        0.1417  0.0990
     3

/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


              precision    recall  f1-score   support

           0       1.00      0.50      0.67         2
           1       1.00      1.00      1.00         2
           2       0.67      1.00      0.80         2
           3       0.67      1.00      0.80         2
           4       1.00      0.50      0.67         2
           5       0.67      1.00      0.80         2
           6       1.00      0.50      0.67         2
           7       0.67      1.00      0.80         2
           8       0.67      1.00      0.80         2
           9       1.00      0.50      0.67         2
          10       1.00      1.00      1.00         2
          11       1.00      1.00      1.00         2
          12       0.00      0.00      0.00         2
          13       1.00      1.00      1.00         2
          14       1.00      0.50      0.67         2
          15       0.40      1.00      0.57         2
          16       1.00      0.50      0.67         2
          17       1.00    